In [2]:
import json
import pandas as pd
import pandas_ta as ta
import numpy as np
from dotenv import load_dotenv
load_dotenv()

True

In [44]:
indicator_list = [] # เก็บข้อมูล indicator ทุกตัวใส่ในนี้
data = pd.read_csv('Oil.csv')
# data.tail()

df = data.copy()
df['timestamp'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None)
df.set_index('timestamp', inplace = True)
df = df.rename(columns={"Open": "open", "High": "high", "Low": "low", "Close": "close", "Volume": "volume"})
df = df.drop(columns=['Date'])
df

C:\Users\A715-72G\AppData\Local\Temp\ipykernel_29292\2556976466.py:6: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  df['timestamp'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None)


,open,high,low,close,Adj Close,volume
timestamp,,,,,,
2016-11-04,39.720001,40.750000,39.250000,40.360001,40.360001,679351.0
2016-12-04,40.349998,42.250000,40.090000,42.169998,42.169998,796837.0
2016-04-13,41.630001,42.419998,41.240002,41.759998,41.759998,730437.0
2016-04-14,41.540001,42.160000,40.840000,41.500000,41.500000,488614.0
2016-04-15,41.430000,41.730000,39.980000,40.360001,40.360001,520146.0
...,...,...,...,...,...,...
2021-02-04,61.450001,61.450001,61.450001,61.450001,61.450001,605567.0
2021-05-04,61.500000,61.500000,57.630001,58.650002,58.650002,438864.0
2021-06-04,58.799999,60.900002,58.619999,59.330002,59.330002,457348.0


#### TODO list
 - ใช้คำสั่ง help(ta.<ชื่อ indicator>) หาข้อมูล input กับ output ของ indicator
 - เช่น macd ใช้คำสั่งตามตัวอย่างข้างล่างนี้

In [45]:
help(ta.macd)

Help on function macd in module pandas_ta.momentum.macd:

macd(close, fast=None, slow=None, signal=None, talib=None, offset=None, **kwargs)
    Moving Average Convergence Divergence (MACD)
    
    The MACD is a popular indicator to that is used to identify a security's trend.
    While APO and MACD are the same calculation, MACD also returns two more series
    called Signal and Histogram. The Signal is an EMA of MACD and the Histogram is
    the difference of MACD and Signal.
    
    Sources:
        https://www.tradingview.com/wiki/MACD_(Moving_Average_Convergence/Divergence)
        AS Mode: https://tr.tradingview.com/script/YFlKXHnP/
    
    Calculation:
        Default Inputs:
            fast=12, slow=26, signal=9
        EMA = Exponential Moving Average
        MACD = EMA(close, fast) - EMA(close, slow)
        Signal = EMA(MACD, signal)
        Histogram = MACD - Signal
    
        if asmode:
            MACD = MACD - Signal
            Signal = EMA(MACD, signal)
          

- จากข้อมูลข้างบนจะได้ macd รับ input เป็น close

``` 
    "name": "MACD",
    "description": "MACD",
    "indicator": "macd",
    "inputs": [
        "close"
    ],
```

- ข้อมูล params ค่าที่ใช้ประกอบด้วย 
    - fast ค่า default = 12
    - slow ค่า default = 26
    - signal ค่า default = 9

```
    "params": [
            {
                "name": "fast",
                "label": "Fast length",
                "type": "number",
                "defaultValue": 12
            },
            {
                "name": "slow",
                "label": "Slow length",
                "type": "number",
                "defaultValue": 26
            },
            {
                "name": "signal",
                "label": "Signal",
                "type": "number",
                "defaultValue": 9
            }
        ],
```

- นำ data มารัน indicator เพื่อหา output

In [46]:
df.ta.macd(fast=12, slow=26, signal=9, append=True)
df

,open,high,low,close,Adj Close,volume,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9
timestamp,,,,,,,,,
2016-11-04,39.720001,40.750000,39.250000,40.360001,40.360001,679351.0,NaN,NaN,NaN
2016-12-04,40.349998,42.250000,40.090000,42.169998,42.169998,796837.0,NaN,NaN,NaN
2016-04-13,41.630001,42.419998,41.240002,41.759998,41.759998,730437.0,NaN,NaN,NaN
2016-04-14,41.540001,42.160000,40.840000,41.500000,41.500000,488614.0,NaN,NaN,NaN
2016-04-15,41.430000,41.730000,39.980000,40.360001,40.360001,520146.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2021-02-04,61.450001,61.450001,61.450001,61.450001,61.450001,605567.0,-0.011614,-0.331051,0.319437
2021-05-04,61.500000,61.500000,57.630001,58.650002,58.650002,438864.0,-0.206204,-0.420513,0.214309
2021-06-04,58.799999,60.900002,58.619999,59.330002,59.330002,457348.0,-0.302066,-0.413100,0.111034


- ข้อมูล outputs ดูจาก column ใหม่ทั้งหมดที่ได้จาก macd มีชื่อประกอบด้วย
    - MACD
    - MACDh
    - MACDs

```
    "outputs": [
            {
                "name": "MACD",
                "label": "macd",
                "graph": "line",
                "type": "number"
            },
            {
                "name": "MACDs",
                "label": "signal",
                "graph": "line",
                "type": "number"
            },
            {
                "name": "MACDh",
                "label": "histogram",
                "graph": "hist",
                "type": "number"
            }
        ]
```

- นำข้อมูลทั้งหมดที่ได้มาสร้างข้อมูล json ตามตัวอย่างข้างล่างนี้
- ตอนรันใช้ Run All เพื่อความสะดวกเพราะ indicator อาจมีเป็นร้อย

In [47]:
macd = {
            "name": "MACD",
            "description": "MACD",
            "indicator": "macd",
            "inputs": [
                "close" # เช็คให้ดีบาง indicator ใช้ volume หรืออืนๆด้วย
            ],
            "params": [
                {
                    "name": "fast",
                    "label": "Fast length", # label คือชื่อไว้แสดงบนหน้าเว็บให้ user อ่าน
                    "type": "number", # type ประกอบด้วย number คือเลข string คืออักษร boolean คือ True, False 
                    "defaultValue": 12
                },
                {
                    "name": "slow",
                    "label": "Slow length",
                    "type": "number",
                    "defaultValue": 26
                },
                {
                    "name": "signal",
                    "label": "Signal",
                    "type": "number",
                    "defaultValue": 9
                }
            ],
            "outputs": [
                {
                    "name": "MACD", # ชื่อนี้จะตรงกับที่อยู่ใน column
                    "label": "macd",
                    "graph": "line", # กรณี plot เป็นเส้น
                    "type": "number"
                },
                {
                    "name": "MACDs", # ชื่อนี้จะตรงกับที่อยู่ใน column
                    "label": "signal",
                    "graph": "line", # กรณี plot เป็นเส้น
                    "type": "number"
                },
                {
                    "name": "MACDh", # ชื่อนี้จะตรงกับที่อยู่ใน column
                    "label": "histogram",
                    "graph": "hist", # กรณี plot เป็นกราฟแท่ง
                    "type": "number"
                }
            ]
        }

indicator_list.append(macd) # เพิ่มข้อมูลเข้า indicator_list
# ทำซ้ำกับ indicator ตัวอื่นๆจนได้ข้อมูลที่ต้องการครบ
# array indicator_list นำไปสร้างไฟล์ json เพื่อนำไปใช้ต่อ

In [48]:
ema =  {
            "name": "EMA",
            "description": "EMA",
            "indicator": "ema",
            "inputs": [
                "close"
            ],
            "params": [
                {
                    "name": "length",
                    "label": "Length",
                    "type": "number",
                    "defaultValue": 50
                }
            ],
            "outputs": [
                {
                    "name": "EMA",
                    "label": "ema",
                    "type": "number"
                }
            ]
        }

indicator_list.append(ema)

In [49]:
rsi = {
            "name": "RSI",
            "description": "RSI",
            "indicator": "rsi",
            "inputs": [
                "close"
            ],
            "params": [
                {
                    "name": "length",
                    "label": "Length",
                    "type": "number",
                    "defaultValue": 14
                }
            ],
            "outputs": [
                {
                    "name": "RSI",
                    "label": "rsi",
                    "type": "number"
                }
            ]
        }

indicator_list.append(rsi)

In [50]:
indicator_list

[{'name': 'MACD',
  'description': 'MACD',
  'indicator': 'macd',
  'inputs': ['close'],
  'params': [{'name': 'fast',
    'label': 'Fast length',
    'type': 'number',
    'defaultValue': 12},
   {'name': 'slow',
    'label': 'Slow length',
    'type': 'number',
    'defaultValue': 26},
   {'name': 'signal', 'label': 'Signal', 'type': 'number', 'defaultValue': 9}],
  'outputs': [{'name': 'MACD',
    'label': 'macd',
    'graph': 'line',
    'type': 'number'},
   {'name': 'MACDs', 'label': 'signal', 'graph': 'line', 'type': 'number'},
   {'name': 'MACDh',
    'label': 'histogram',
    'graph': 'hist',
    'type': 'number'}]},
 {'name': 'EMA',
  'description': 'EMA',
  'indicator': 'ema',
  'inputs': ['close'],
  'params': [{'name': 'length',
    'label': 'Length',
    'type': 'number',
    'defaultValue': 50}],
  'outputs': [{'name': 'EMA', 'label': 'ema', 'type': 'number'}]},
 {'name': 'RSI',
  'description': 'RSI',
  'indicator': 'rsi',
  'inputs': ['close'],
  'params': [{'name': 'l

In [51]:
jsonString = json.dumps(indicator_list)
jsonFile = open("IndyData.json", "w")
jsonFile.write(jsonString)
jsonFile.close()